In [ ]:
# repo-root bootstrap: notebooks live in notebooks/, code lives one level up
import sys, pathlib
_root = pathlib.Path.cwd()
if not (_root / "spphot_eval.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

In [2]:
astra = pd.read_parquet('/home/100/mj8805/scr_mk27/bulge-ages-and-orbits/data/astra_photogeo.parquet')
len(astra)

1712317

In [10]:
from pathlib import Path
cluster_dir = Path('/home/100/mj8805/scr_mk27/distance-estimator/clusters/clusters/catalogues')
data = [f for f in cluster_dir.glob('*.txt')]

In [14]:
cluster_data = {}

for pth in data:
    cluster_name = pth.stem
    cluster_data[cluster_name] = pd.read_csv(pth, sep='\s+', header=None, low_memory=False)

In [ ]:
from tqdm import tqdm

cluster_counts = {}

for cluster_name, df in tqdm(cluster_data.items()):
    # If the cluster file has a header, adjust header reading in previous cell
    # Assuming column 0 is source_id
    if df.shape[1] > 0 and 'gaia_dr3_source_id' in astra.columns:
        cluster_source_ids = set(df.iloc[:, 0].astype(str))
        astra_source_ids = set(astra['gaia_dr3_source_id'].astype(str))
        n_crossmatched = len(cluster_source_ids & astra_source_ids)
        print(f"Cluster {cluster_name}: {n_crossmatched}/{len(cluster_source_ids)} crossmatched rows")
        cluster_counts[cluster_name] = n_crossmatched

In [19]:
# Sort cluster_counts by n_crossmatched (the number of crossmatched rows)
sorted_cluster_counts = sorted(cluster_counts.items(), key=lambda item: item[1], reverse=True)
for cluster_name, n_crossmatched in sorted_cluster_counts[:5]:
    print(f"{cluster_name}: {n_crossmatched}")

NGC_5139_oCen: 1663
NGC_6656_M_22: 511
NGC_104_47Tuc: 431
NGC_3201: 378
NGC_6121_M_4: 323


In [22]:
from pathlib import Path

output_dir = Path("/home/100/mj8805/scr_mk27/distance-estimator/clusters/matched_astra_parquets")
output_dir.mkdir(parents=True, exist_ok=True)

for cluster_name, _ in tqdm(sorted_cluster_counts[:10]):
    df = cluster_data[cluster_name]
    # Gather cluster source_ids as string for matching
    cluster_source_ids = df.iloc[:, 0].astype(str)
    # Match against astra's gaia_dr3_source_id
    matched = astra[astra['gaia_dr3_source_id'].astype(str).isin(cluster_source_ids)]
    # Save to parquet
    output_path = output_dir / f"{cluster_name}_astra_matched.parquet"
    matched.to_parquet(output_path, index=False)
    print(f"Saved {len(matched)} matched rows for {cluster_name} to {output_path}")

 10%|██████████████▊                                                                                                                                     | 1/10 [00:01<00:12,  1.41s/it]

Saved 1786 matched rows for NGC_5139_oCen to /home/100/mj8805/scr_mk27/distance-estimator/clusters/matched_astra_parquets/NGC_5139_oCen_astra_matched.parquet


 20%|█████████████████████████████▌                                                                                                                      | 2/10 [00:02<00:10,  1.29s/it]

Saved 561 matched rows for NGC_6656_M_22 to /home/100/mj8805/scr_mk27/distance-estimator/clusters/matched_astra_parquets/NGC_6656_M_22_astra_matched.parquet


 30%|████████████████████████████████████████████▍                                                                                                       | 3/10 [00:03<00:08,  1.18s/it]

...

Saved 253 matched rows for NGC_5904_M_5 to /home/100/mj8805/scr_mk27/distance-estimator/clusters/matched_astra_parquets/NGC_5904_M_5_astra_matched.parquet


 90%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏              | 9/10 [00:08<00:00,  1.35it/s]

Saved 180 matched rows for NGC_2808 to /home/100/mj8805/scr_mk27/distance-estimator/clusters/matched_astra_parquets/NGC_2808_astra_matched.parquet


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.15it/s]

Saved 330 matched rows for NGC_5272_M_3 to /home/100/mj8805/scr_mk27/distance-estimator/clusters/matched_astra_parquets/NGC_5272_M_3_astra_matched.parquet
